# GBPUSD/EURUSD — independent-day strategies

Four strategies on the cointegration spread, fit and back-tested **independently per trading day** on three months: **2024-08, 2024-09, 2025-08**.

### Pipeline (paper §2–§4)
1. **Pre-averaging** (paper eq 2.2): bid/ask ticks of leg A and leg B are synchronised at the tick level (union + ffill), then every L=500 consecutive synchronised ticks are collapsed into one bar by averaging the mids. Same L, same boundaries on both legs → pre-averaged legs perfectly co-time.
2. **Rolling cointegration** (paper §3.1): hedge ratio $\beta_t$ by rolling OLS with window $W_\beta=25$ on the pre-averaged log prices; spread $S_t = \log P^A_t - \beta_t \log P^B_t - \alpha_t$.
3. **Rolling $z$-score** (paper §3.2): $Z_t$ over $W_z=15$ bars.
4. **MS-AR(1) on the spread** (paper §4.3): EM via Baum–Welch, $K=2$, multi-seed init.
5. **Regime labelling by innovation variance** (paper §4.7): MR = regime with smallest $\sigma^{(k)}$ (quiet, safe), DR = regime with largest $\sigma^{(k)}$ (volatile, danger).
6. **Four strategies**: Buy & Hold, Baseline (z-score), AR (binary gate $\mathbf{1}\{\gamma_t^{MR} \geq 1-\delta\}$), MS-AR (dynamic threshold $\gamma_t^{MR} z_q + \gamma_t^{DR} z_v$).

The OOS forward filter is intentionally deferred — we use the in-sample smoothed posteriors $\gamma_t^{MR}$ directly inside the strategies, so this is a *best-case* picture of how the regime information would be used if it were known.


In [6]:
# Colab bootstrap (silently skipped on local environments).
import importlib.util, sys, os
if importlib.util.find_spec('google.colab') is not None:
    import subprocess
    subprocess.run(
        ['curl', '-sL',
         'https://raw.githubusercontent.com/egil10/stk-mat2011/main/code/scripts/colab.py',
         '-o', '/content/colab.py'],
        check=True,
    )
    sys.path.insert(0, '/content')
    from colab import setup
    setup('code/strats')
else:
    sys.path.insert(0, os.path.abspath('../scripts'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CWD           : /content/stk-mat2011/code/strats
Scripts path  : /content/stk-mat2011/code/scripts
Data symlink  : /content/stk-mat2011/code/data/processed -> /content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed
Parquet files : 916


In [7]:
%pip install --quiet arch statsmodels numba optuna


In [8]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

PAIR_A, PAIR_B = 'GBPUSD', 'EURUSD'
MONTHS         = ['202408', '202409', '202508']
DATA_DIR       = '../data/processed'

# Tick-synced pre-averaged bars (paper eq 2.2).
# L=500 -> each bar is the mean of 500 SYNCHRONISED ticks on both legs.
# Tradeoff vs L=200: 2.5x fewer bars/day -> 2.5x fewer trades -> 2.5x less
# cumulative cost erosion; alpha per trade is unchanged.
BAR_CFG = dict(
    agg_type     = 'tick',
    threshold    = 500,       # block size L
    price_agg    = 'mean',    # 'mean' triggers tick-sync + pre-averaging
    active_hours = (0, 24),
)

# Per-day fit. Rolling windows scaled down to keep ~30 bars of warmup,
# leaving the bulk of each day available for trading at the new L=500.
FIT_CFG = dict(
    coint_window  = 25,    # W_beta  (was 50 at L=200)
    z_window      = 15,    # W_z     (was 25 at L=200)
    k_regimes     = 2,
    winsorize_std = 4.0,
    scaling       = 10000,
    n_init        = 3,
)

# Strategies. z_q widened to 2.0 to cut entry rate roughly in half
# (the binding constraint at this bar resolution is cost-per-trade).
STRAT_CFG = dict(
    z_quiet          = 2.0,    # was 1.3
    z_volatile       = 3.0,    # widened in tandem with z_quiet
    exit_z           = 0.0,
    danger_threshold = 0.30,
    fee_bps          = 0.5,
    slippage_mode    = 'half_spread',
    flatten_eod      = True,
    prob_smoothing   = 0,
)

ANN_FACTOR = 252 * 24 * 60   # tick-clock annualisation


## Per-month run

For each month we

1. Aggregate raw bid/ask ticks into L-tick pre-averaged bars (`SPREAD`).
2. Fit each trading day independently — rolling cointegration ($\beta_t, \alpha_t$), rolling $z$-score, MS-AR(1) on the day's spread (`ENGINE.each_day`). Regimes labelled by $\sigma$.
3. Run Buy & Hold, Baseline, AR(1) gate, MS-AR(1) dynamic-threshold through the same back-test machinery (`BACKTESTER`).
4. Score with `TEARSHEET`.


In [9]:
def files(pair, month):
    p = pair.lower()
    return (
        [f'{DATA_DIR}/{p}_dukascopy_ask_{month}.parquet'],
        [f'{DATA_DIR}/{p}_dukascopy_bid_{month}.parquet'],
    )


def fmt_pnl(bt):
    return {s: bt[f'Return_{s}'].fillna(0).sum() * 1e4
            for s in ['BuyHold', 'Baseline', 'AR', 'MS_AR']}


runs = {}
for m in MONTHS:
    print(f'\n=== {m}  {PAIR_A}/{PAIR_B} ===')

    ask_a, bid_a = files(PAIR_A, m)
    ask_b, bid_b = files(PAIR_B, m)

    df = SPREAD(**BAR_CFG).build([ask_a, bid_a, ask_b, bid_b], verbose=True)
    n_bars = len(df)
    n_days = df.index.normalize().unique().shape[0]
    print(f'  {n_bars:,} pre-averaged bars over {n_days} trading days')

    fitted, params = ENGINE.each_day(df, **FIT_CFG, verbose=True)
    bt = BACKTESTER(fitted).run(**STRAT_CFG)
    runs[m] = (bt, params, fitted)

    pnl = fmt_pnl(bt)
    bh, base, ar, ms = pnl['BuyHold'], pnl['Baseline'], pnl['AR'], pnl['MS_AR']
    print(
        f'  net pnl (bps):  BH={bh:+8.1f}  Base={base:+8.1f}  '
        f'AR={ar:+8.1f}  MS-AR={ms:+8.1f}'
    )



=== 202408  GBPUSD/EURUSD ===
built 7113 pre-averaged bars (L=500 synced ticks/block, 3,557,484 synced ticks total)
  7,113 pre-averaged bars over 22 trading days
Each-day fit | coint_window=25 | z_window=15 | k_regimes=2 | winsor=4.0σ | scale=x10000 | n_init=3 | days=22
  -> 22/22 days fitted (0 too short, 0 HMM failures)
  net pnl (bps):  BH=   +65.1  Base=  -647.6  AR=  -123.0  MS-AR=  -133.4

=== 202409  GBPUSD/EURUSD ===
built 7353 pre-averaged bars (L=500 synced ticks/block, 3,677,253 synced ticks total)
  7,353 pre-averaged bars over 21 trading days
Each-day fit | coint_window=25 | z_window=15 | k_regimes=2 | winsor=4.0σ | scale=x10000 | n_init=3 | days=21
  -> 21/21 days fitted (0 too short, 0 HMM failures)
  net pnl (bps):  BH=    +4.3  Base=  -671.5  AR=  -181.8  MS-AR=  -223.5

=== 202508  GBPUSD/EURUSD ===
built 5517 pre-averaged bars (L=500 synced ticks/block, 2,759,476 synced ticks total)
  5,517 pre-averaged bars over 21 trading days
Each-day fit | coint_window=25 | z_w

## Per-month tearsheets

Combined dashboards print to the cell output. Every panel is **also** saved as its own PDF so individual figures are ready to drop into the thesis. PDFs land in `plots/<PAIR>/` (Drive on Colab, `./plots/` locally).


In [ ]:
from pathlib import Path
from plotting import default_pdf_dir

PAIR_DIR = default_pdf_dir() / f'{PAIR_A}_{PAIR_B}'
PAIR_DIR.mkdir(parents=True, exist_ok=True)
print(f'PDFs -> {PAIR_DIR}')

for m, (bt, params, _) in runs.items():
    print(f'{"="*28}  {m}  {"="*28}')
    try:
        ts = TEARSHEET(
            bt,
            df_params  = params,
            save_pdf   = True,
            pdf_dir    = str(PAIR_DIR),
            pdf_prefix = f'{PAIR_A}_{PAIR_B}_{m}',
        )
        ts.generate_report()

        # Render the combined dashboards FIRST so BuyHold + all panels show
        # even if the per-panel PDF saves below blow up for any reason.
        try:
            ts.plot_performance()
        except Exception as e:
            print(f'  [plot_performance failed: {type(e).__name__}: {e}]')
        try:
            ts.plot_positions_and_regimes()
        except Exception as e:
            print(f'  [plot_positions_and_regimes failed: {type(e).__name__}: {e}]')
        try:
            ts.plot_markov_dynamics()
        except Exception as e:
            print(f'  [plot_markov_dynamics failed: {type(e).__name__}: {e}]')

        # Then save each panel as its own PDF (newer tearsheet only).
        # Wrapped separately so a missing method on an older tearsheet
        # cannot suppress the dashboards above.
        if hasattr(ts, 'save_individual_perf_panels'):
            try:
                ts.save_individual_perf_panels()
            except Exception as e:
                print(f'  [save individual perf panels failed: {type(e).__name__}: {e}]')
        if hasattr(ts, 'save_individual_markov_panels'):
            try:
                ts.save_individual_markov_panels()
            except Exception as e:
                print(f'  [save individual markov panels failed: {type(e).__name__}: {e}]')
    except Exception as e:
        print(f'  [{m} fully failed: {type(e).__name__}: {e}]')

## Cross-month summary

PnL in bps, Sharpe annualised on the tick-clock (`252 × 24 × 60` bars/year, matches `MONTH` default), and one round-trip per `Target` flip cycle.


In [11]:
def annualised_sharpe(returns):
    r = returns.fillna(0)
    sd = r.std()
    return float(r.mean() / sd * np.sqrt(ANN_FACTOR)) if sd > 0 else 0.0


rows = []
for m, (bt, _, _) in runs.items():
    row = {'Month': m}
    for s in ['BuyHold', 'Baseline', 'AR', 'MS_AR']:
        r = bt[f'Return_{s}']
        row[f'{s}_PnL_bps'] = float(r.fillna(0).sum() * 1e4)
        row[f'{s}_Sharpe']  = annualised_sharpe(r)
        row[f'{s}_Trades']  = int((bt[f'Target_{s}'].diff().abs() > 0).sum() / 2)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Month')

print('\n=== PnL bps by month ===')
print(summary[[c for c in summary.columns if c.endswith('PnL_bps')]].round(1).to_string())

print('\n=== Sharpe (tick-clock annualised) ===')
print(summary[[c for c in summary.columns if c.endswith('Sharpe')]].round(2).to_string())

print('\n=== Round-trip trade counts ===')
print(summary[[c for c in summary.columns if c.endswith('Trades')]].to_string())

summary



=== PnL bps by month ===
        BuyHold_PnL_bps  Baseline_PnL_bps  AR_PnL_bps  MS_AR_PnL_bps
Month                                                               
202408             65.1            -647.6      -123.0         -133.4
202409              4.3            -671.5      -181.8         -223.5
202508             49.9            -571.2      -128.4         -152.2

=== Sharpe (tick-clock annualised) ===
        BuyHold_Sharpe  Baseline_Sharpe  AR_Sharpe  MS_AR_Sharpe
Month                                                           
202408            3.32           -54.72     -30.03        -25.06
202409            0.28           -67.44     -47.97        -42.68
202508            3.93           -73.27     -43.92        -32.82

=== Round-trip trade counts ===
        BuyHold_Trades  Baseline_Trades  AR_Trades  MS_AR_Trades
Month                                                           
202408               0              281         79            82
202409               0              

,BuyHold_PnL_bps,BuyHold_Sharpe,BuyHold_Trades,Baseline_PnL_bps,Baseline_Sharpe,Baseline_Trades,AR_PnL_bps,AR_Sharpe,AR_Trades,MS_AR_PnL_bps,MS_AR_Sharpe,MS_AR_Trades
Month,,,,,,,,,,,,
202408,65.086104,3.316811,0,-647.609603,-54.717532,281,-123.030442,-30.027578,79,-133.426405,-25.058754,82
202409,4.316203,0.275594,0,-671.493634,-67.439235,268,-181.788624,-47.968787,73,-223.485473,-42.683822,76
202508,49.893815,3.928720,0,-571.234490,-73.274077,176,-128.445659,-43.919239,49,-152.155178,-32.823303,46


### Per-day regime parameters

Each row in `runs[m][1]` is the per-day fit summary: bar count, $\beta$, regime $\sigma$, regime $\rho$, regime means and transition probabilities. Useful for sanity-checking the HMM and for the paper's tables.


In [12]:
for m, (_, params, _) in runs.items():
    print(f'\n=== {m}  per-day fits ===')
    print(params.round(4).to_string())



=== 202408  per-day fits ===
            Bars    Beta   Alpha  MR_Sigma  DR_Sigma  MR_Rho  DR_Rho   MR_Mu   DR_Mu  P_MR_MR  P_DR_DR
Date                                                                                                  
2024-08-01   394  1.6327  0.1176    0.0001    0.0002  0.8290  0.9467 -0.0007 -0.0039   0.9586      NaN
2024-08-02   432  0.9772  0.1620    0.0001    0.0002  0.8384  0.9306  0.0001 -0.0002   0.9356      NaN
2024-08-05   814 -0.9185  0.3295    0.0001    0.0003  0.9197  0.8952  0.0002 -0.0005   0.9821      NaN
2024-08-06   509  1.3385  0.1195    0.0001    0.0002  0.8304  0.9092 -0.0005 -0.0008   0.9868      NaN
2024-08-07   370 -0.8033  0.3092    0.0000    0.0001 -0.9765  0.9158  0.0000  0.0006   0.0001      NaN
2024-08-08   347  0.6766  0.1833    0.0001    0.0001  0.2750  0.9456 -0.0003  0.0003   0.7872      NaN
2024-08-09   218  0.8656  0.1676    0.0000    0.0001 -0.1250  0.9435  0.0000  0.0014   0.0000      NaN
2024-08-12   218  0.1936  0.2271    0.0001 